# Time-Series Demand & Revenue Forecasting (Prophet)
This model forecasts `Units` and `Revenue` 30-90 days into the future for the AI Dashboard.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Load prepared data
df = pd.read_csv('ml_ready_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

## Aggregate Daily Global Demand

In [ ]:
daily_demand = df.groupby('Date').agg({'Units': 'sum', 'Revenue': 'sum'}).reset_index()

# Prepare for Prophet (Revenue)
df_prophet = daily_demand[['Date', 'Revenue']].rename(columns={'Date': 'ds', 'Revenue': 'y'})

# Train-Test Split (Last 30 days as test)
train = df_prophet[:-30]
test = df_prophet[-30:]

## Train Prophet Model (Revenue)

In [ ]:
model = Prophet(yearly_seasonality=True, weekly_seasonality=True)
model.fit(train)

future = model.make_future_dataframe(periods=60) # Predict 30 days of test + 30 days future
forecast = model.predict(future)

## Evaluate & Visualize

In [ ]:
forecast_test = forecast[['ds', 'yhat']].tail(60).head(30)
mape = mean_absolute_percentage_error(test['y'], forecast_test['yhat'])
print(f"Revenue Forecast MAPE: {mape:.2%}")

fig1 = model.plot(forecast)
plt.title("Revenue Forecast")
plt.show()

## Export Dashboard Data

In [ ]:
# Predict future 60 days from max date
model_full = Prophet(yearly_seasonality=True, weekly_seasonality=True)
model_full.fit(df_prophet)
future_full = model_full.make_future_dataframe(periods=60)
forecast_full = model_full.predict(future_full)

# Save to CSV for Power BI
forecast_output = forecast_full[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(60)
forecast_output = forecast_output.rename(columns={'ds': 'Date', 'yhat': 'Predicted_Revenue', 'yhat_lower': 'Pessimistic_Revenue', 'yhat_upper': 'Optimistic_Revenue'})
forecast_output.to_csv('predicted_revenue_dashboard.csv', index=False)
print("Exported 'predicted_revenue_dashboard.csv'")